Author : Yolan Ankaine

Date : May 2026

# **Cooling to Bose-Einstein Condensate**

## **Black-Box Bayesian Optimization (3BO)** 

The experimental system consists of the atomic vapour interacting with the 4 stage cooling experimental setup (i.e., MOT, CMOT, PGC, CDT). The whole system can be treated as a black-box function that takes in an input parameter vector $ \bold{x} \in \mathcal{X} $ and maps that onto the experimental observation $ \bold{y} \in \mathcal{Y} $: 

$$
f : \mathcal{X} \rightarrow \mathcal{Y}
$$

The GP surrogate model never assumes an explicit form of $f$ and instead, learns it entirely from the data taken from absorption/TOF measurements.

In [ ]:
# import ML libraries
import os 
import sys
import json       # for pre-processing
import warnings
from botorch.exceptions.warnings import InputDataWarning, NumericsWarning, OptimizationWarning

# import standard libraries 
import random
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# import functions 
from optimisation.generate_dataset import generate_initial_data
from optimisation.bayesian_opt_funcs import BO_function,plot_performance, plot_gp_slice
from src.globals import get_cooling_stages,  get_param_bounds
from src.utils.get_results import get_final_params
from src.solvers.artiq_interface import BOARTIQInterface
from src.utils.timer import TicTocGenerator, tic_toc 




Housekeeping

In [ ]:

#Initialize the generator
TicToc = TicTocGenerator() # create an instance of the TicTocGen generator
#Call tic_toc("tic") to start timing and tic_toc("toc") to get the elapsed time.
tic_toc(action="tic", print_time=True)         # starts timer

# Close all plots 
plt.close('all')
# Clear the console 
                        # You can print new lines as a workaround to clear the output
print("\033c", end="")  # ANSI escape sequence to clear the console on some 

sys.path.append(os.path.abspath('..'))  # go up one level to project root

# save changes automatically 
%load_ext autoreload
%autoreload 2

# get the global variables and parameters
param_bounds = get_param_bounds()
cooling_stages = get_cooling_stages()
param_bounds = get_param_bounds()
param_names = list(param_bounds.keys())

## **Preliminaries**

Define fixed physical parameters for the experimental setup.

In [ ]:
coil_params = {
   " N_turns" : 30,         # number of turns in each coil (20 to 50)
    "R_coil":  30 * 1E-3,   # radius of each coil (25 to 40 mm)
    "d_coil":  50 * 1E-3,   # distance between the two coils (30 to 60 mm)
}

### **Session configuration**

In [ ]:
# Create the Artiq Interface
artiq = BOARTIQInterface(host="::1")

# Define the session configuration and results file path
config_path = "C:/home/ae19663/artiq/ARTIQ/repository/bayesian_optimization/BEC_BO/outputs/results"        # location for saved configuration
save_path = config_path # modify later

# Obtain session configuration
# session_config = read_session_config(config_path)
session_config = artiq.read_session_config(config_path)

# Define simulation parameters 
n_init = session_config['n_init']               # no.of initial samples to generate
m_samples = session_config['m_samples']         # no.of repeated shots per experimental run
n_iter = session_config['n_iterations']         # max no.of iterations in the BO loop

n_points = 1                                    # no.of points to explore
user_seed = 42                                  # initial seed for reproducibility
n_calls = 0                                     # no.of calls to the BO func during optimization

# Retrieve the session optimization stages + parameters
selected_stages = session_config['selected_stages'] 
active_params = session_config['active_params']

# Obtain the physical and unit bounds 
phys_bounds = session_config['phys_bounds'] 
unit_bounds = session_config['unit_bounds']

session_name = session_config['session_name']   # session name 

# parameter to track 

## **Complete Optimization**

In [ ]:

# 1. Generate initial data set 
x, init_x, init_y, init_y_var, best_init_y = generate_initial_data(n_init, m_samples,save_path,
                                                                active_params,selected_stages,
                                                                user_seed, artiq)
# make a copy of initial data (in physical units)
x_active = x.clone()

# 2. Select the acquisition function type
acq_func_type = "logEI"

acq_funcs = ["EI", "logEI", "UCB", "PI"]

colours = plt.rcParams["axes.prop_cycle"].by_key()["color"]  # ['#1f77b4', '#ff7f0e', ...]
# Create mapping: string → colour
colour_map = {item: colours[i % len(colours)] for i, item in enumerate(acq_funcs)}

# ignore warnings 
with warnings.catch_warnings():
    warnings.simplefilter("ignore", InputDataWarning)
    warnings.simplefilter("ignore", NumericsWarning)
    warnings.simplefilter("ignore", RuntimeWarning)
    warnings.simplefilter("ignore", OptimizationWarning)

    # --------------------  3. BO loop --------------------
    for i in range(n_iter):
            init_x, init_y, best_init_y, n_calls, model = BO_function(
                                                    init_x, init_y, init_y_var, best_init_y,
                                                    phys_bounds, unit_bounds, n_points, 
                                                    acq_func_type, n_init, m_samples, 
                                                    i, active_params, artiq, session_name)
            
            # track the performance : model convergence 
            perc_imp = plot_performance(init_y, best_init_y)
            
            # plot confidence interval with GP slice
            plot_gp_slice(model, init_x, init_y, init_y_var,
                  param_idx=25, param_name="t_evap",
                  output_idx=0, output_name="N",
                  n_test=200, z=1.96)
            
print(f"Model converges to {perc_imp[-1]:.5f} % improvement")

# obtain the final model parameters (at best values)
final_results = get_final_params(init_x, init_y, best_init_y, session_config)

### **Results**

In [ ]:
print(f"The final results of {session_name} is:")
print(final_results)

# add the results to the session configuration
key = "final results"
session_config[key] = final_results 

# save to original location 
with open(config_path, "w") as f:
        json.dump(session_config, f, indent=4)


---
## **STEP 1: Initialization**

Generate the initial dataset to fit the **GP** using either
1. Experimental data 
2. Simulated data

Since experimental data is scarce, we go for option 2. Can use [Latin Hypercube Sampling (LHS)](https://en.wikipedia.org/wiki/Latin_hypercube_sampling): 
$$ 
\mathcal{D}_0 = \{ (\bold{x_1}, \bold{y_1}), (\bold{x_2}, \bold{y_2}), \dots , (\bold{x_{n_{init}}}, \bold{y_{n_{init}}}) \}
$$

LHS is a quasi-random search method that uses a space-filling design to guarantee much better space coverage than pure random sampling with the same number of points. It does this by 
- Dividing each parameter dimension into $n$ equal intervals
- Places exactly one sample in each interval
- Randomly permutes the assignments across dimensions so no two points share a row or column in any 2D projection 

This avoids the systematic gaps of grid search and the large random pockets of random search.

## **STEP 2: Select optimization parameters**

Let the user decide which parameters to optimize for by selecting from the available $d$ parameters. 

## **STEP 3 : Build the  Surrogate Model**

We choose a surrogate model such as a Gaussian Proces (GP).

**Bounds for the surrogate model**.

BO is solving a constrained optimization problem, so the bounds 
- Define the input domain
- Tell BO, "only search here, nowhere else"
- Give the kernel a physical length scale
- Encode physics amd hardware constraints


## **Step 4 : Implement an Acquisition Function (AF)**

Examples of acquisition functions include **expected improvement (EI)**, **upper confidence bound (UCB)**, and **probability of improvement (PI)**.  

Here we will use **log(EI)** which, 
- measures how much the objective function is expected to increase relative to the current best value.
- is numerically stable log-transform of EI. Avoids underflow in low-error regimes where EI → 0.

EI is defined as, 
$$EI(x) = 𝔼 [max (f(x) - f(x^+),0)].$$

where $f(x^+)$ is the current best observed value of $f$.

This can be implement with the ``LogExpectedImprovement`` function from [Monte Carlo Acquisition Functions](https://botorch.org/docs/acquisition/).

### **Evaluate and Maximize the AF**

Find the next sampling point by maximizing $AF(x)$ such that, 
$$ x_{next} = \arg\max_{x} \, AF(x)  $$

Use $x_{next}$ to evaluate $f(x)$ and add this new datapoint to the dataset. Finally, update the GP model

>  To maximize $AF(x)$ use the [``optimize_acqf``](https://botorch.org/docs/optimization/) function to perform a joint optimization by default, or sequential optimization when ``sequential=True``.



